<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/890_MSOv3_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is an **excellent core intelligence module**. It clearly implements the architectural idea you designed earlier:

```
Telemetry → Intelligence Metrics → Signals
```

In other words, this file is the **intelligence engine of the orchestrator**.


---

# Signature Metrics Engine — Mission Orchestrator V3

This module converts raw operational telemetry into **actionable intelligence metrics**.

While the telemetry loader gathers factual data about mission activity, the metrics layer transforms that data into **operational signals leaders can interpret quickly**.

This step is crucial because large volumes of telemetry are difficult to reason about directly. Executives and operators need **clear indicators of system health**, not raw logs.

The functions in this module compute four standardized metrics:

```
Mission Efficiency Score
Operational Friction Score
Agent Reliability Score
Human Dependency Index
```

These metrics act as the **vital signs of the AI operational system**.

---

# Role in the Orchestrator Architecture

This module sits between **telemetry ingestion** and **executive reporting**.

The workflow now looks like this:

```
Telemetry Loader
      ↓
Signature Metrics Engine   ← this module
      ↓
Operational Status Signals
      ↓
Executive Report Generator
```

By separating these steps, the system remains **transparent and testable**.

Each stage performs one clear responsibility:

* telemetry loader gathers operational data
* metrics engine calculates intelligence signals
* reporting layer communicates results

This separation is one of the reasons the system remains **predictable and auditable**.

---

# KPI Selection Logic

The `_get_primary_kpi` function determines which KPI should be used to evaluate mission performance.

Because missions can define multiple KPIs, the orchestrator must identify the **primary metric** for efficiency scoring.

The function follows a clear hierarchy:

1. If the configured default KPI exists, use it.
2. Otherwise fall back to the first KPI defined for the mission.

This ensures the orchestrator can still compute mission efficiency even if configuration is incomplete.

Importantly, the function normalizes KPI data into a consistent structure containing:

```
kpi_name
baseline
target
actual
higher_is_better
```

Standardizing the KPI format prevents downstream metric calculations from becoming fragile.

---

# Mission Efficiency Score

The Mission Efficiency Score measures **how effectively a mission achieves its target KPI**.

The function handles several real-world scenarios:

• missions with explicit targets
• missions without targets but with baselines
• KPIs where higher values are better
• KPIs where lower values are better

The score is normalized to a **0–100 range**, making it easy to interpret and compare across missions.

Example interpretation:

```
95 → mission nearly meeting target
70 → mission underperforming
40 → mission significantly off target
```

This metric answers the most fundamental operational question:

**Is the mission achieving the outcome it was designed to produce?**

---

# Operational Friction Score

The Operational Friction Score measures **process inefficiencies caused by human delays or workflow bottlenecks**.

It is calculated using:

```
human_delay_minutes
total mission runtime
```

The result represents the percentage of mission time lost to friction.

For example:

```
18 minutes human delay
52 minute mission runtime
```

Produces:

```
Operational Friction Score = 34
```

This metric is extremely useful operationally because it highlights where automation opportunities exist.

Executives often want to know:

```
Where are humans slowing the system down?
```

This metric provides that signal.

---

# Agent Reliability Score

The Agent Reliability Score measures how consistently agents perform their tasks without escalation or failure.

The score combines two important operational factors:

```
success rate
escalation rate
```

Success rate contributes **60% of the score**, while low escalation rates contribute **40%**.

Agents with insufficient task volume are excluded from the calculation.

This safeguard prevents misleading conclusions when sample sizes are too small.

The resulting score represents the **average reliability of agents participating in the mission workflow**.

For operational leaders, this metric answers a key question:

```
Are the agents stable enough to trust in production?
```

---

# Human Dependency Index

The Human Dependency Index measures **how much the mission workflow depends on human intervention**.

The system attempts to calculate this from mission-level telemetry:

```
human_interventions / total_tasks
```

If that information is not available, it falls back to task-level telemetry.

Specifically, it counts how many tasks required human approval.

This fallback logic ensures the orchestrator can still compute automation metrics even when mission-level summaries are incomplete.

The resulting score communicates how automated the system really is.

Example interpretation:

```
10 → highly automated workflow
35 → moderate human involvement
70 → heavily manual system
```

For organizations investing in AI automation, this metric is especially important.

It helps leadership determine whether AI investments are actually **reducing operational workload**.

---

# Data Quality Signal

Before producing executive insights, the system evaluates the **quality of the telemetry itself**.

The `_compute_data_quality` function checks:

```
number of mission runs
presence of KPI configuration
presence of anomalous runs
```

Based on these checks, the orchestrator assigns a signal:

```
HIGH
MEDIUM
LOW
```

This signal appears in the final report alongside a brief explanation.

For example:

```
DATA QUALITY: HIGH
(8 runs loaded; no missing KPIs)
```

Providing this signal improves trust in the system’s conclusions.

Operational leaders need to know **whether they should trust the analysis**.

---

# Mission Status Signal

After computing metrics, the orchestrator converts them into a single operational signal.

Possible values include:

```
HEALTHY
AT RISK
OFF TARGET
```

This signal is derived from deterministic rules:

• efficiency below threshold → OFF TARGET
• high friction or low reliability → AT RISK
• critical risk event → AT RISK override

This rule-based approach is intentional.

Many AI systems attempt to determine system health using model reasoning.

This system instead uses **explicit business thresholds**, which makes the logic easy to audit and modify.

For operational environments, deterministic decision rules often inspire **greater confidence than probabilistic model outputs**.

---

# Final Metric Assembly

The `compute_signature_metrics` function orchestrates all calculations.

It pulls telemetry from the orchestrator state, computes the intelligence metrics, determines the operational signals, and writes the results back into the state.

The following values are produced:

```
mission_efficiency_score
operational_friction_score
agent_reliability_score
human_dependency_index
mission_status_signal
data_quality_signal
data_quality_details
```

These values then become the foundation for the **executive report** generated later in the workflow.

---

# Why This Design Matters Operationally

This metrics engine demonstrates an important design philosophy:

**AI systems should produce measurable operational intelligence.**

Rather than relying on opaque model reasoning, the orchestrator calculates explicit metrics from structured telemetry.

This design has several advantages:

• transparent decision logic
• reproducible results
• configurable thresholds
• clear operational signals
• auditable system behavior

For organizations deploying AI in operational roles, these characteristics are essential.

They allow AI systems to be evaluated and trusted in the same way traditional operational systems are.

---

# Small Improvements Worth Considering

The implementation is already strong. A few small improvements could further improve robustness.

---

### Initialize Safe Float Conversion

Some fields are cast directly to `float`. A small helper function could reduce repeated try/except blocks.

Example pattern:

```
safe_float(value, default=0.0)
```

This would simplify metric calculations.

---

### Track Metric Confidence

You could optionally add a field indicating which metrics were successfully computed.

Example:

```
metrics_available = {
    efficiency: True
    friction: True
    reliability: False
}
```

This can improve reporting transparency.

---

### Improve Anomaly Detection

Currently the anomaly detection placeholder is:

```
has_anomalies = False
```

In the future, deterministic checks could detect:

• unusually long mission runtimes
• abnormal task durations
• extreme KPI deviations

These checks would strengthen the **data quality signal**.

---

# Overall Evaluation

This module successfully transforms operational telemetry into **clear intelligence metrics**.

Strengths include:

• deterministic metric calculations
• robust fallback logic for incomplete data
• safeguards against small sample sizes
• configurable thresholds through configuration
• clear separation between telemetry and reporting

This metrics engine forms the **analytical core of the Mission Orchestrator**.

Once these signals are produced, the reporting layer can translate them into **decision-ready operational insights**.


In [ ]:
from __future__ import annotations

from typing import Any, Dict, List, Optional

from config import MissionOrchestratorV3Config, MissionOrchestratorV3State


def _get_primary_kpi(
    mission_kpis: Dict[str, Any],
    default_name: str,
) -> Optional[Dict[str, Any]]:
    if not mission_kpis:
        return None
    if default_name in mission_kpis:
        value = mission_kpis[default_name]
        if isinstance(value, dict):
            return value
        return {
            "kpi_name": default_name,
            "baseline": None,
            "target": None,
            "actual": value,
            "higher_is_better": False,
        }
    # Fallback: take first entry
    for name, value in mission_kpis.items():
        if isinstance(value, dict):
            return value
        return {
            "kpi_name": name,
            "baseline": None,
            "target": None,
            "actual": value,
            "higher_is_better": False,
        }
    return None


def _compute_efficiency_score(primary_kpi: Dict[str, Any]) -> Optional[float]:
    target = primary_kpi.get("target")
    actual = primary_kpi.get("actual")
    baseline = primary_kpi.get("baseline")
    higher_is_better = bool(primary_kpi.get("higher_is_better"))

    if actual is None:
        return None

    if target is not None:
        try:
            if higher_is_better:
                ratio = float(actual) / float(target) if target else None
            else:
                ratio = float(target) / float(actual) if actual else None
        except (TypeError, ZeroDivisionError):
            ratio = None
    else:
        # No target: compare vs baseline if available
        if baseline is None:
            return None
        try:
            if higher_is_better:
                ratio = float(actual) / float(baseline) if baseline else None
            else:
                ratio = float(baseline) / float(actual) if actual else None
        except (TypeError, ZeroDivisionError):
            ratio = None

    if ratio is None:
        return None

    score = max(0.0, min(100.0, ratio * 100.0))
    return round(score, 1)


def _compute_operational_friction(
    mission_runs: List[Dict[str, Any]],
    task_execution_logs: List[Dict[str, Any]],
) -> Optional[float]:
    if not mission_runs or not task_execution_logs:
        return None

    total_runtime = 0.0
    human_delay = 0.0

    for run in mission_runs:
        try:
            total_runtime += float(run.get("total_duration_minutes", 0.0))
        except (TypeError, ValueError):
            continue

    for task in task_execution_logs:
        try:
            human_delay += float(task.get("human_delay_minutes", 0.0))
        except (TypeError, ValueError):
            continue

    if total_runtime <= 0:
        return None

    fraction = max(0.0, min(1.0, human_delay / total_runtime))
    return round(fraction * 100.0, 1)


def _compute_agent_reliability(
    agent_performance_stats: List[Dict[str, Any]],
    min_tasks_for_agent_metrics: int,
) -> Optional[float]:
    if not agent_performance_stats:
        return None

    scores: List[float] = []

    for agent in agent_performance_stats:
        n_tasks = agent.get("tasks_executed") or agent.get("task_count")
        try:
            n = int(n_tasks)
        except (TypeError, ValueError):
            n = 0

        if n < min_tasks_for_agent_metrics:
            continue

        success_rate = float(agent.get("success_rate", 0.0))
        escalation_rate = float(agent.get("escalation_rate", 0.0))
        low_escalation = max(0.0, 100.0 - escalation_rate)
        score = (success_rate * 0.6) + (low_escalation * 0.4)
        scores.append(score)

    if not scores:
        return None

    avg = sum(scores) / len(scores)
    return round(avg, 1)


def _compute_human_dependency(
    mission_runs: List[Dict[str, Any]],
    task_execution_logs: List[Dict[str, Any]],
) -> Optional[float]:
    if not mission_runs and not task_execution_logs:
        return None

    total_tasks = 0
    human_interventions = 0

    for run in mission_runs:
        try:
            total_tasks += int(run.get("total_tasks", 0))
            human_interventions += int(run.get("human_interventions", 0))
        except (TypeError, ValueError):
            continue

    if not total_tasks:
        # Fallback: derive from task_execution_logs
        total_tasks = len(task_execution_logs)
        human_interventions = 0
        for task in task_execution_logs:
            if task.get("human_approval_required"):
                human_interventions += 1

    if total_tasks <= 0:
        return None

    fraction = max(0.0, min(1.0, human_interventions / float(total_tasks)))
    return round(fraction * 100.0, 1)


def _compute_data_quality(
    run_count: int,
    mission_runs: List[Dict[str, Any]],
    mission_kpis: Dict[str, Any],
    min_runs_for_trend: int,
) -> (str, str):
    if run_count <= 0:
        return "LOW", "No mission runs available."

    missing_kpis = not bool(mission_kpis)
    has_anomalies = False  # Placeholder: deterministic anomaly checks can be added later.

    if run_count >= min_runs_for_trend and not missing_kpis and not has_anomalies:
        level = "HIGH"
    elif missing_kpis or has_anomalies:
        level = "LOW"
    else:
        level = "MEDIUM"

    details_parts = [f"{run_count} runs loaded"]
    if missing_kpis:
        details_parts.append("missing KPI config")
    if has_anomalies:
        details_parts.append("anomalies detected")

    details = "; ".join(details_parts)
    return level, details


def _compute_mission_status(
    efficiency: Optional[float],
    friction: Optional[float],
    reliability: Optional[float],
    human_dependency: Optional[float],
    has_critical_risk: bool,
) -> str:
    if has_critical_risk:
        return "AT RISK"

    # Simple, deterministic rule set
    if efficiency is not None and efficiency < 70.0:
        return "OFF TARGET"

    if (
        (friction is not None and friction > 50.0)
        or (reliability is not None and reliability < 75.0)
        or (human_dependency is not None and human_dependency > 40.0)
    ):
        return "AT RISK"

    return "HEALTHY"


def compute_signature_metrics(
    state: MissionOrchestratorV3State,
    config: MissionOrchestratorV3Config,
) -> MissionOrchestratorV3State:
    """Compute V3 signature intelligence metrics and signals."""
    mission_runs = state.get("mission_runs") or []
    task_execution_logs = state.get("task_execution_logs") or []
    mission_risk_events = state.get("mission_risk_events") or []
    agent_performance_stats = state.get("agent_performance_stats") or []
    mission_kpis = state.get("mission_kpis") or {}
    run_count = state.get("run_count") or 0

    primary_kpi = _get_primary_kpi(mission_kpis, config.default_primary_kpi)
    efficiency = _compute_efficiency_score(primary_kpi) if primary_kpi else None
    friction = _compute_operational_friction(mission_runs, task_execution_logs)
    reliability = _compute_agent_reliability(
        agent_performance_stats, config.min_tasks_for_agent_metrics
    )
    human_dep = _compute_human_dependency(mission_runs, task_execution_logs)

    has_critical_risk = any(
        (e.get("severity") == "high" or e.get("severity") == "critical")
        for e in mission_risk_events or []
    )

    data_quality, data_quality_details = _compute_data_quality(
        run_count=run_count,
        mission_runs=mission_runs,
        mission_kpis=mission_kpis,
        min_runs_for_trend=config.min_runs_for_trend,
    )

    mission_status = _compute_mission_status(
        efficiency=efficiency,
        friction=friction,
        reliability=reliability,
        human_dependency=human_dep,
        has_critical_risk=has_critical_risk,
    )

    state["mission_efficiency_score"] = efficiency
    state["operational_friction_score"] = friction
    state["agent_reliability_score"] = reliability
    state["human_dependency_index"] = human_dep
    state["mission_status_signal"] = mission_status
    state["data_quality_signal"] = data_quality
    state["data_quality_details"] = data_quality_details

    return state

